# 05e — Kiểm chứng các con số trong báo cáo

Notebook này tồn tại để **mọi con số trong Mục 7.3.4 và 7.3.6 của báo cáo đều dựng lại được**.

Trước đây phần sàn mẫu số `eps_elev` trích một số con số mà không ghi phạm vi dòng, nên
không ai kiểm lại được. Notebook khai báo phạm vi tường minh ở ô dưới, rồi tính lại toàn bộ.

Chạy xong sẽ ghi ra `data/model/v4/05e_kiem_chung/con_so_bao_cao.json` để báo cáo trích dẫn.

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

GOC = Path("../..")
VAO = GOC / "data/model/v4/05_selected"
RA  = GOC / "data/model/v4/05e_kiem_chung"
RA.mkdir(parents=True, exist_ok=True)

EPS = 0.05           # san mau so, dung dung gia tri trong config
BUOC_PHUT = 15       # tam du bao H1

COT = ["site_id", "timestamp", "sin_elevation", "energy_generated_kwh",
       "is_daylight", "energy_source", "exclude_from_training"]

## 1. Phạm vi

Một dòng được tính khi **nhãn của nó tại `T+15` phút** là số đo thật, thuộc ban ngày, và
dòng đó không bị loại khỏi huấn luyện. Đây đúng là tập dòng đi vào hàm mục tiêu, nên con số
rút ra mới nói được điều gì về ảnh hưởng của `eps_elev`.

In [3]:
tr = pd.read_parquet(VAO / "v4_train_selected.parquet", columns=COT)
tr = tr.sort_values(["site_id", "timestamp"])

g = tr.groupby("site_id")
tr["nhan_y"]   = g["energy_generated_kwh"].shift(-1)
tr["nhan_day"] = g["is_daylight"].shift(-1)
tr["nhan_src"] = g["energy_source"].shift(-1)

hop_le = (
    tr["nhan_y"].notna()
    & tr["nhan_day"].fillna(False).astype(bool)
    & tr["nhan_src"].astype(str).eq("measured")
    & ~tr["exclude_from_training"].fillna(False).astype(bool)
)
d = tr[hop_le].copy()
print(f"Tap train            : {len(tr):,} dong")
print(f"Pham vi tinh          : {len(d):,} dong")

Tap train            : 1,550,856 dong
Pham vi tinh          : 635,564 dong


## 2. Vành `0 < sin h <= eps` bỏ đi bao nhiêu

`eps_elev` không quyết định độ chính xác, nó quyết định **số dòng còn lại**. Ô này đo đúng
phần bị vành cắt mất.

In [4]:
vanh  = d[(d["sin_elevation"] > 0) & (d["sin_elevation"] <= EPS)]
ngoai = d[d["sin_elevation"] > EPS]
tong_kwh = float(d["nhan_y"].sum())

so = {
    "pham_vi": "train, nhan tai T+15 phut la so do that ban ngay, khong bi loai",
    "eps_elev": EPS,
    "goc_nang_do": round(float(np.degrees(np.arcsin(EPS))), 2),
    "so_dong": int(len(d)),
    "so_dong_giu_lai": int(len(ngoai)),
    "ty_le_giu_lai_pct": round(len(ngoai) / len(d) * 100, 2),
    "tong_san_luong_kwh": round(tong_kwh, 0),
    "trong_vanh_so_dong": int(len(vanh)),
    "trong_vanh_kwh": round(float(vanh["nhan_y"].sum()), 0),
    "trong_vanh_tb_kwh_moi_buoc": round(float(vanh["nhan_y"].mean()), 4),
    "ngoai_vanh_tb_kwh_moi_buoc": round(float(ngoai["nhan_y"].mean()), 4),
    "ty_le_san_luong_trong_vanh": round(float(vanh["nhan_y"].sum()) / tong_kwh, 6),
}
so["ty_le_san_luong_trong_vanh_pct"] = round(so["ty_le_san_luong_trong_vanh"] * 100, 3)

for k, v in so.items():
    print(f"{k:34s} : {v}")

pham_vi                            : train, nhan tai T+15 phut la so do that ban ngay, khong bi loai
eps_elev                           : 0.05
goc_nang_do                        : 2.87
so_dong                            : 635564
so_dong_giu_lai                    : 629167
ty_le_giu_lai_pct                  : 98.99
tong_san_luong_kwh                 : 4240381.0
trong_vanh_so_dong                 : 5378
trong_vanh_kwh                     : 2279.0
trong_vanh_tb_kwh_moi_buoc         : 0.4238
ngoai_vanh_tb_kwh_moi_buoc         : 6.7349
ty_le_san_luong_trong_vanh         : 0.000537
ty_le_san_luong_trong_vanh_pct     : 0.054


## 3. Trung bình mỗi trạm mỗi ngày có bao nhiêu bước

In [5]:
moi_ngay  = d.assign(ngay=d["timestamp"].dt.date).groupby(["site_id", "ngay"]).size()
vanh_ngay = vanh.assign(ngay=vanh["timestamp"].dt.date).groupby(["site_id", "ngay"]).size()

so["tb_buoc_moi_tram_moi_ngay"]      = round(float(moi_ngay.mean()), 1)
so["tb_buoc_trong_vanh_moi_tram_ngay"] = round(float(vanh_ngay.sum()) / len(moi_ngay), 2)
print(f"TB buoc/tram/ngay            : {so['tb_buoc_moi_tram_moi_ngay']}")
print(f"TB buoc trong vanh/tram/ngay : {so['tb_buoc_trong_vanh_moi_tram_ngay']}")

TB buoc/tram/ngay            : 40.8
TB buoc trong vanh/tram/ngay : 0.35


## 4. Cỡ tập kiểm định

Báo cáo từng ghi *"phân vị 99 của k trên tập kiểm định (≈289.000 giá trị)"* mà không nói
289.000 lấy ở đâu. Ô này đếm ra con số thật.

In [6]:
val = pd.read_parquet(VAO / "v4_val_selected.parquet", columns=COT)
ban_ngay_do_that = (
    val["is_daylight"].fillna(False).astype(bool)
    & val["energy_source"].astype(str).eq("measured")
)
so["val_tong_so_dong"] = int(len(val))
so["val_ban_ngay_do_that"] = int(ban_ngay_do_that.sum())
print(f"Tap kiem dinh              : {so['val_tong_so_dong']:,} dong")
print(f"  ban ngay va do that      : {so['val_ban_ngay_do_that']:,} dong")

Tap kiem dinh              : 723,114 dong
  ban ngay va do that      : 307,342 dong


## 5. Ghi kết quả để báo cáo trích dẫn

In [7]:
dich = RA / "con_so_bao_cao.json"
dich.write_text(json.dumps(so, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Da ghi: {dich}")
print(json.dumps(so, ensure_ascii=False, indent=2))

Da ghi: ../../data/model/v4/05e_kiem_chung/con_so_bao_cao.json
{
  "pham_vi": "train, nhan tai T+15 phut la so do that ban ngay, khong bi loai",
  "eps_elev": 0.05,
  "goc_nang_do": 2.87,
  "so_dong": 635564,
  "so_dong_giu_lai": 629167,
  "ty_le_giu_lai_pct": 98.99,
  "tong_san_luong_kwh": 4240381.0,
  "trong_vanh_so_dong": 5378,
  "trong_vanh_kwh": 2279.0,
  "trong_vanh_tb_kwh_moi_buoc": 0.4238,
  "ngoai_vanh_tb_kwh_moi_buoc": 6.7349,
  "ty_le_san_luong_trong_vanh": 0.000537,
  "ty_le_san_luong_trong_vanh_pct": 0.054,
  "tb_buoc_moi_tram_moi_ngay": 40.8,
  "tb_buoc_trong_vanh_moi_tram_ngay": 0.35,
  "val_tong_so_dong": 723114,
  "val_ban_ngay_do_that": 307342
}
